In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import scripts.common as cmn
import scripts.weight as wght

In [5]:
wght.inspect_weights()

,model,original,trained
0,yolov8n,False,False
1,yolo26n,True,True


In [7]:
wght.download_weights()

,status,size (MiB),file
model,,,
yolov8n,Downloaded,6.25,yolov8n.pt
yolo26n,Already exists,5.29,yolo26n.pt


## Реестр и выбор моделей

`cmn.MODELS` остаётся единственным словарём включения и отключения моделей.
`cmn.MODELS_REGESTRY` содержит метаданные реализаций с теми же ключами.
Для семейства необходимо явно указать файл весов: размер модели автоматически
не выбирается. При скачивании используется имя файла из реестра, которое может
отличаться от локального идентификатора модели.

Реализован только бэкенд Ultralytics. Записи MMDetection и RF-DETR зарезервированы
для будущих адаптеров. Остальные реализации отключены и снабжены комментариями.
При обучении неподдерживаемые записи пропускаются с указанием причины;
некорректный датасет или ошибка обучения останавливают выполнение.


In [ ]:
from scripts.contracts import DatasetSpec, TrainConfig, ModelArtifacts
from scripts.workflows import train_models, load_model

# Текущий датасет содержит разметку для детекции. Для сегментации экземпляров
# укажите YAML датасета YOLO с полигональной разметкой и task="segment".
dataset = DatasetSpec(cmn.DATA_YAML, task="detect")
training_config = TrainConfig(
    epochs=5,
    image_size=640,
    batch=1,
    device="cpu",  # При необходимости замените на "cuda:0" или "mps".
    seed=0,
    options={"workers": 0, "plots": True},
)


In [ ]:
# Установите True для обучения всех включённых моделей, поддерживающих обучение.
RUN_TRAINING = False
training_batch = None
if RUN_TRAINING:
    training_batch = train_models(dataset, training_config)
    print("Завершено:", list(training_batch.runs))
    print("Пропущено:", training_batch.skipped)
else:
    print("Обучение отключено. Установите RUN_TRAINING=True для запуска примера на пять эпох.")


## Загрузка результата конкретного запуска и предсказание

Каждый запуск сохраняет лучшие и последние веса, а также `manifest.json`
с описанием модели, версией пакета, фактическими настройками обучения,
хешами датасета и выборок и метриками. Копия в `weights/trained/<model-id>.pt`
содержит веса последнего успешного запуска; для воспроизводимости используйте
артефакт конкретного эксперимента.

Общий интерфейс предсказания принимает путь к локальному изображению, изображение
PIL или массив NumPy формы H×W×3 с типом `uint8` и порядком каналов **BGR**.
Рамки и маски используют координаты исходного изображения. Исходные результаты
провайдера доступны для визуализации.


In [ ]:
MODEL_TO_INSPECT = "yolo26n"
IMAGE_PATH = cmn.DATASET_PATH / "images" / "00a357bc-17.jpg"

if training_batch is not None and MODEL_TO_INSPECT in training_batch.runs:
    run = training_batch.runs[MODEL_TO_INSPECT]
    predictor = load_model(MODEL_TO_INSPECT, run.best_artifacts, device=training_config.device)
    predictions = predictor.predict(IMAGE_PATH, imgsz=640, conf=0.25)
    for prediction in predictions:
        print("Рамки:", prediction.boxes_xyxy)
        print("Классы:", prediction.class_ids)
        import matplotlib.pyplot as plt
        plt.figure(figsize=(10, 7))
        plt.imshow(prediction.native.plot()[..., ::-1])  # BGR -> RGB
        plt.axis("off")
        plt.show()
else:
    print("Сначала обучите выбранную модель или явно загрузите веса существующего запуска.")

# Для загрузки предыдущего запуска укажите точный путь к его весам:
# artifacts = ModelArtifacts(cmn.RUNS_DIR / "<run-name>" / "weights" / "best.pt")
# predictor = load_model(MODEL_TO_INSPECT, artifacts, device=training_config.device)


## Необязательный пример предсказания по подсказкам

Предсказание по подсказкам не зависит от обучения. Адаптер предоставляет
предсказание для SAM, MobileSAM, SAM 2, SAM 3 и FastSAM, но не заявляет
поддержку их обучения через Ultralytics. В нормализованном результате маскам
без семантических классов не назначаются вымышленные идентификаторы классов.

Примеры ниже используют явные спецификации и не изменяют `MODELS`.
По умолчанию выполнение отключено, поскольку файлы весов могут быть большими.
Для SAM 3 может потребоваться ручное получение весов с разрешением доступа:
перед использованием поместите `sam3.pt` в `cmn.ORIGINAL_WEIGHTS_DIR`.


In [ ]:
from dataclasses import replace
from scripts.backends import get_backend

RUN_PROMPT_EXAMPLE = False
# Явно задаём вариант семейства SAM; для других семейств используется тот же подход.
prompt_spec = replace(cmn.MODELS_REGESTRY["sam"], checkpoint="sam_b.pt")
if RUN_PROMPT_EXAMPLE:
    backend = get_backend(prompt_spec)
    artifacts = backend.download(prompt_spec)
    prompted = backend.load(prompt_spec, artifacts, device=training_config.device)
    masks = prompted.predict(IMAGE_PATH, prompts={"points": [[100, 100]], "labels": [1]})
    masks[0].native.show()

# Другие виды подсказок (координаты должны соответствовать выбранному изображению):
# YOLOWorld: prompts={"text": ["tobacco", "weed"]}
# YOLOE: prompts={"text": ["tobacco"]}
# YOLOE: prompts={"visual_prompts": {"bboxes": [[10, 10, 100, 100]], "cls": [0]}}
# FastSAM: prompts={"text": ["leaf"]} или {"bboxes": [[10, 10, 100, 100]]}
# SAM/MobileSAM/SAM2/SAM3, интерактивный режим: prompts={"points": [[100, 100]], "labels": [1]}
# SAM3, поиск по понятиям: prompts={"text": ["tobacco"]} или {"exemplars": [[10, 10, 100, 100]]}
# Сопровождение объектов на видео с сохранением состояния не входит в этот API.
